In [ ]:
# Install once if required
# %pip install ultralytics matplotlib torch

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from ultralytics import YOLO

torch.manual_seed(42)
np.random.seed(42)

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 1. Load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")
print("Pretrained YOLOv8n model loaded.")

# 2. Train with first 10 layers frozen
model.train(
    data="coco128.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    freeze=10,
    device=device,
    workers=0,
    project="assignment6_results",
    name="stage1_frozen",
    exist_ok=True,
    plots=True
)

stage1_dir = Path(model.trainer.save_dir)
stage1_weights = stage1_dir / "weights" / "best.pt"
if not stage1_weights.exists():
    stage1_weights = stage1_dir / "weights" / "last.pt"

print("Frozen-layer training completed:", stage1_weights)

# 3. Fine-tune all layers
model = YOLO(str(stage1_weights))
model.train(
    data="coco128.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    freeze=0,
    lr0=0.001,
    device=device,
    workers=0,
    project="assignment6_results",
    name="stage2_finetune",
    exist_ok=True,
    plots=True
)

stage2_dir = Path(model.trainer.save_dir)
final_weights = stage2_dir / "weights" / "best.pt"
if not final_weights.exists():
    final_weights = stage2_dir / "weights" / "last.pt"

print("Fine-tuning completed:", final_weights)

# 4. Evaluate final model
model = YOLO(str(final_weights))
metrics = model.val(
    data="coco128.yaml",
    imgsz=640,
    batch=8,
    device=device,
    workers=0
)

print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP@50:", round(metrics.box.map50, 4))
print("mAP@50-95:", round(metrics.box.map, 4))

# 5. Detect objects in a sample image
image_url = "https://ultralytics.com/images/bus.jpg"
results = model.predict(
    source=image_url,
    conf=0.25,
    imgsz=640,
    device=device
)

annotated = results[0].plot()
annotated = annotated[:, :, ::-1]  # BGR to RGB

plt.figure(figsize=(10, 7))
plt.imshow(annotated)
plt.title("Object Detection using Transfer Learning")
plt.axis("off")
plt.tight_layout()
plt.show()

# 6. Display detected object names, confidence and bounding boxes
boxes = results[0].boxes
if boxes is not None and len(boxes) > 0:
    for box in boxes:
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        coordinates = box.xyxy[0].tolist()
        print(
            f"Object: {model.names[class_id]} | "
            f"Confidence: {confidence:.2%} | "
            f"Bounding box: {[round(v, 2) for v in coordinates]}"
        )
else:
    print("No objects detected.")

print("Practical 6 completed.")
